In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("SHOW SCHEMAS").show()

In [ ]:
# Imports
import json
import uuid
from datetime import datetime, timezone

import requests

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

In [ ]:
# Configuration
VAULT_URL = "https://mbta-api-key.vault.azure.net/"
SECRET_NAME = "mbta-api-key"
MBTA_BASE_URL = "https://api-v3.mbta.com/predictions"
ROUTE_FILTER = "Red"

In [ ]:
# Get the API key
def get_api_key():
    return notebookutils.credentials.getSecret(VAULT_URL, SECRET_NAME)

In [ ]:
# Fetch predictions
def fetch_predictions(api_key):
    params = {"filter[route]": ROUTE_FILTER, "include": "schedule"}
    headers = {"x-api-key": api_key}
    resp = requests.get(MBTA_BASE_URL, params=params, headers=headers)
    resp.raise_for_status()
    return resp.json()

In [ ]:
# Bronze schema
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, BooleanType, TimestampType
)

def get_bronze_schema():
    return StructType([
        StructField("prediction_id", StringType(), True),
        StructField("route_id", StringType(), True),
        StructField("trip_id", StringType(), True),
        StructField("stop_id", StringType(), True),
        StructField("vehicle_id", StringType(), True),
        StructField("direction_id", IntegerType(), True),
        StructField("stop_sequence", IntegerType(), True),
        StructField("predicted_arrival_time", StringType(), True),
        StructField("predicted_departure_time", StringType(), True),
        StructField("arrival_uncertainty", IntegerType(), True),
        StructField("departure_uncertainty", IntegerType(), True),
        StructField("last_trip", BooleanType(), True),
        StructField("revenue", StringType(), True),
        StructField("schedule_relationship", StringType(), True),
        StructField("status", StringType(), True),
        StructField("trip_headsign", StringType(), True),
        StructField("update_type", StringType(), True),
        StructField("schedule_id", StringType(), True),
        StructField("scheduled_arrival_time", StringType(), True),
        StructField("scheduled_departure_time", StringType(), True),
        StructField("scheduled_drop_off_type", IntegerType(), True),
        StructField("scheduled_pickup_type", IntegerType(), True),
        StructField("scheduled_timepoint", BooleanType(), True),
        StructField("raw_json", StringType(), True),
        StructField("ingested_at", TimestampType(), True),
    ])

In [ ]:
# Flatten the API response into Bronze rows
def flatten_predictions(api_response):
    predictions = api_response.get("data", [])
    included = api_response.get("included", [])
    schedules_by_id = {item["id"]: item for item in included if item["type"] == "schedule"}
    ingested_at = datetime.now(timezone.utc)

    rows = []
    for pred in predictions:
        attrs = pred["attributes"]
        rels = pred["relationships"]
        schedule_ref = rels["schedule"]["data"]
        schedule_id = schedule_ref["id"] if schedule_ref else None
        matched_schedule = schedules_by_id.get(schedule_id) if schedule_id else None
        sched_attrs = matched_schedule["attributes"] if matched_schedule else {}

        rows.append({
            "prediction_id": pred["id"],
            "route_id": rels["route"]["data"]["id"] if rels["route"]["data"] else None,
            "trip_id": rels["trip"]["data"]["id"] if rels["trip"]["data"] else None,
            "stop_id": rels["stop"]["data"]["id"] if rels["stop"]["data"] else None,
            "vehicle_id": rels["vehicle"]["data"]["id"] if rels["vehicle"]["data"] else None,
            "direction_id": attrs.get("direction_id"),
            "stop_sequence": attrs.get("stop_sequence"),
            "predicted_arrival_time": attrs.get("arrival_time"),
            "predicted_departure_time": attrs.get("departure_time"),
            "arrival_uncertainty": attrs.get("arrival_uncertainty"),
            "departure_uncertainty": attrs.get("departure_uncertainty"),
            "last_trip": attrs.get("last_trip"),
            "revenue": attrs.get("revenue"),
            "schedule_relationship": attrs.get("schedule_relationship"),
            "status": attrs.get("status"),
            "trip_headsign": attrs.get("trip_headsign"),
            "update_type": attrs.get("update_type"),
            "schedule_id": schedule_id,
            "scheduled_arrival_time": sched_attrs.get("arrival_time"),
            "scheduled_departure_time": sched_attrs.get("departure_time"),
            "scheduled_drop_off_type": sched_attrs.get("drop_off_type"),
            "scheduled_pickup_type": sched_attrs.get("pickup_type"),
            "scheduled_timepoint": sched_attrs.get("timepoint"),
            "raw_json": json.dumps({"prediction": pred, "schedule": matched_schedule}),
            "ingested_at": ingested_at,
        })
    return rows

In [ ]:
# Write to Bronze (append)
def write_to_bronze(rows, schema, table_name):
    df = spark.createDataFrame(rows, schema=schema)
    df.write.mode("append").saveAsTable(table_name)
    return len(rows)

In [ ]:
# Audit log schema
def get_pipeline_runs_schema():
    return StructType([
        StructField("run_id", StringType(), True),
        StructField("pipeline_run_id", StringType(), True),
        StructField("stage_name", StringType(), True),
        StructField("status", StringType(), True),
        StructField("rows_affected", IntegerType(), True),
        StructField("started_at", TimestampType(), True),
        StructField("completed_at", TimestampType(), True),
        StructField("error_message", StringType(), True),
    ])

In [ ]:
# Log a run
def log_run(stage_name, status, rows_affected, started_at, error_message=None):
    log_row = [{
        "run_id": str(uuid.uuid4()),
        "pipeline_run_id": None,
        "stage_name": stage_name,
        "status": status,
        "rows_affected": rows_affected,
        "started_at": started_at,
        "completed_at": datetime.now(timezone.utc),
        "error_message": error_message
    }]
    df = spark.createDataFrame(log_row, schema=get_pipeline_runs_schema())
    df.write.mode("append").synapsesql("WH_MBTA.control.pipeline_runs")

In [ ]:
# Get the API key
mbta_api_key = get_api_key()
print("Key retrieved successfully:", mbta_api_key is not None)

In [ ]:
# Run the full ingestion
started_at = datetime.now(timezone.utc)
try:
    api_response = fetch_predictions(mbta_api_key)
    rows = flatten_predictions(api_response)
    row_count = write_to_bronze(rows, get_bronze_schema(), "bronze.predictions_raw")
    log_run("bronze_predictions_ingest", "SUCCESS", row_count, started_at)
    print(f"Bronze predictions ingest succeeded: {row_count} rows")
except Exception as e:
    log_run("bronze_predictions_ingest", "FAILED", 0, started_at, str(e))
    print(f"Bronze predictions ingest failed: {e}")
    raise